In [ ]:
!pip install datasets &> log.txt #датасеты из huggingface

In [ ]:
!pip install evaluate &> log.txt #метрики из huggingface

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data.dataloader import Dataset, DataLoader
from torch import nn

from torch.nn.utils.rnn import pad_packed_sequence, pack_padded_sequence
import sys
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from torch.optim import Adam
import os
import subprocess # позволяет запускать системные программы в Python

import matplotlib.pyplot as plt
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter #рисует loss/accuracy, learning_rate и т.д
import shutil
import random
import itertools
import evaluate
bleu_metric = evaluate.load('bleu')

In [ ]:
VALIDATION_PROP = 0.1
RANDOM_STATE = 79

X_TRAIN_PATH = 'en_train.txt'
Y_TRAIN_PATH = 'ru_train.txt'
X_VALID_PATH = 'en_valid.txt'
Y_VALID_PATH = 'ru_valid.txt'
TRAIN = False

HIDDEN_SIZE = 256
LSTM_LAYERS = 3
INPUT_SIZE = 256
BATCH_SIZE = 32
IS_BIDIRECTIONAL_LAYERS = False
CNT_CLASSES = None

SOS_TOKEN = "<SOS>" # start of sentence token
EOS_TOKEN = "<EOS>" # end of sentence token
UNK_TOKEN = "<UNK>" # unknown token
PAD_TOKEN = "<PAD>" # padding token
PAD_IND = 3
SOS_IND = 0
EOS_IND = 1
EPOCHS = 10
MODEL_NAME = 'lstm_basic_model'
SCHEDULER_LAMBDA_PARAM = 0.95
device = 'cuda' if torch.cuda.is_available() else 'cpu'

MODEL_FOLDER_PATH = MODEL_NAME
LR = 0.0004
CLIP = 1
TEACHER_FORCING_RATIO = 0
FULL_DATA_PATH = 'data.txt'
DOWNLOAD_DATA_URL = 'https://www.dropbox.com/s/yy2zqh34dyhv07i/data.txt?dl=1'

WRITE_CNT_STEPS = 100
SCHEDULER_CNT_STEPS = 300
GREEDY_DECODING_LABEL = 'greedy'
BEAM_SEARCH_DECODING_LABEL = 'beam_search'

NEC_TOKENS_LEN = 100  # tokens max length for inference
MAX_TEXT_LEN = 100 # max symbols length for text
BEAM_WIDTH = 3

signs = '.,?!:; '
numbers = ''.join([str(i) for i in range(10)])
RUSSIAN_LETTERS_AND_SIGNS = 'абвгдеёжзийклмнопрстуфхцчшщъыьэюя' + signs + numbers
ENGLISH_LETTERS_AND_SIGNS = ''.join([chr(i) for i in range(ord('a'), ord('z') + 1)]) + signs + numbers
print(RUSSIAN_LETTERS_AND_SIGNS, ENGLISH_LETTERS_AND_SIGNS)

абвгдеёжзийклмнопрстуфхцчшщъыьэюя.,?!:; 0123456789 abcdefghijklmnopqrstuvwxyz.,?!:; 0123456789


В качестве токенов будем использовать буквы

In [ ]:
subprocess.check_call(['wget', DOWNLOAD_DATA_URL, '-O', FULL_DATA_PATH])

0

In [ ]:
# напишем функцию для очистки текста
def clean_text(text):
    text = text.lower().strip()
    delaem = [t for t in text if t in RUSSIAN_LETTERS_AND_SIGNS or t in ENGLISH_LETTERS_AND_SIGNS]
    res_text = ''.join(delaem)
    return res_text

In [ ]:
# пишем свой токенизатор для работы с символами и простыми словами
class Tokenizer:
    def __init__(self, texts, eos, unk, pad, sos, tokens=None):
        self.sos = sos
        self.eos = eos
        self.unk = unk
        self.pad = pad
        self.tokens = tokens

        self.create_tokenizer(texts)

    def create_tokenizer(self, texts):
        texts = [clean_text(text) for text in texts]

        tokens = set('\n'.join(list(texts)).replace('\n', ''))
        tokens = [token for token in tokens if token not in (self.sos, self.eos, self.unk, self.pad) and len(token)]

        self.tokens = self.tokens or [self.sos, self.eos, self.unk, self.pad] + tokens

        self.token2idx = {token: id for id, token in enumerate(self.tokens)}
        self.idx2token = {id: token for id, token in enumerate(self.tokens)}
        self.sos_idx = self.token2idx[self.sos]
        self.eos_idx = self.token2idx[self.eos]
        self.unk_idx = self.token2idx[self.unk]
        self.pad_idx = self.token2idx[self.pad]



    def tokenize(self, texts):
        tokens = []
        for tok in texts:
            if tok in self.token2idx:
                tokens.append(tok)
            else:
                tokens.append(self.unk)
        return [self.sos] + tokens + [self.eos]

    def convert_tokens_to_idx(self, tokens):
        idx_list = [self.token2idx.get(token, self.unk_idx) for token in tokens]
        return idx_list

    def convert_idx_to_tokens(self, idx_list):
        tokens = [self.idx2token.get(idx) for idx in idx_list]
        return tokens

    def convet_text_to_idx(self, text, seq_len):
        tokens = self.tokenize(text)[:seq_len]
        return self.convert_tokens_to_idx(tokens)


In [ ]:
# создаем токенайзер объект класса токенайзер
def prepare_tokenizer(texts_path):
    with open(texts_path) as f:
        lines = [clean_text(line.strip()) for line in f.readlines()]

    return Tokenizer(lines, sos=SOS_TOKEN, eos=EOS_TOKEN, unk=UNK_TOKEN, pad=PAD_TOKEN)

In [ ]:
# загружаем данные и делим на train и validation
with open(FULL_DATA_PATH) as f:
    split_lines = [line.strip().split('\t') for line in f.readlines()]

en_texts = [line[0] for line in split_lines]
ru_texts = [line[1] for line in split_lines]

en_train_texts, en_val_texts, ru_train_texts, ru_val_texts = train_test_split(en_texts, ru_texts, test_size=VALIDATION_PROP, random_state=RANDOM_STATE, shuffle=True)

# фильтруем данные с помощьб длины русского языка
with open(X_TRAIN_PATH, 'w') as f:
    t1 = [text for i, text in enumerate(en_train_texts) if len(ru_train_texts[i]) < MAX_TEXT_LEN]
    f.write('\n'.join(t1))

with open(Y_TRAIN_PATH, 'w') as f:
    t2 = [text for i,text in enumerate(ru_train_texts) if len(text) < MAX_TEXT_LEN]
    f.write('\n'.join(t2))

with open(X_VALID_PATH, 'w') as f:
    t3 = [text for i, text in enumerate(en_val_texts) if len(ru_val_texts[i]) < MAX_TEXT_LEN]
    f.write('\n'.join(t3))

with open(Y_VALID_PATH, 'w') as f:
    t4 = [text for i,text in enumerate(ru_val_texts) if len(text) < MAX_TEXT_LEN]
    f.write('\n'.join(t4))


In [ ]:
x_tokenizer = prepare_tokenizer(X_TRAIN_PATH)
y_tokenizer = prepare_tokenizer(Y_TRAIN_PATH)

In [ ]:
from enum import nonmember
class TranslationDataset:
    def __init__(self,
                 x_seq_path=None,
                 x_tokenizer=None,
                 x_seq_len=None,
                 y_seq_path=None,
                 y_tokenizer=None,
                 y_seq_len=None,
                 is_train=True):
        self.is_train = is_train
        with open(x_seq_path) as f:
            self.x_seq_list = [line.strip() for line in f.readlines()]

        self.x_tokenizer = x_tokenizer or prepare_tokenizer(x_seq_path)
        self.x_seq_len = x_seq_len

        if self.is_train:
            with open(y_seq_path) as f:
                self.y_seq_list = [line.strip() for line in f.readlines()]
            self.y_tokenizer = y_tokenizer or prepare_tokenizer(y_seq_path)
        else:
            self.y_seq_list = None
            self.y_tokenizer = None
        self.y_seq_len = y_seq_len

    def __getitem__(self, idx):
        x = clean_text(self.x_seq_list[idx])
        x = self.x_tokenizer.convet_text_to_idx(x, seq_len=self.x_seq_len)
        x_pad = [self.x_tokenizer.pad_idx] * max(0, self.x_seq_len - len(x))
        x = x + x_pad
        x = torch.tensor(x)

        if self.is_train:
            y = clean_text(self.y_seq_list[idx])
            y = self.y_tokenizer.convet_text_to_idx(y, seq_len=self.y_seq_len)
            y_pad = [self.y_tokenizer.pad_idx] * max(0, self.y_seq_len - len(y))
            y = y + y_pad
            y = torch.tensor(y)
            return x,y
        return x

    def __len__(self):
        return len(self.x_seq_list)

    def cnt_y_tokens(self):
        return len(self.y_tokenizer.tokens)

    def cnt_x_tokens(self):
        return len(self.x_tokenizer.tokens)

Наконец-то создаём Encoder + Decoder Модель

In [ ]:
class EncoderDecoderModel(nn.Module):
    def __init__(self, input_voc_size, cnt_classes, device=None, input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE, lstm_layers=LSTM_LAYERS, is_bidirectional = IS_BIDIRECTIONAL_LAYERS):
        super(EncoderDecoderModel, self).__init__()
        self.cnt_classes = cnt_classes
        self.input_size=input_size
        self.hidden_size = hidden_size
        self.lstm_layers = lstm_layers

        self.out_features = self.cnt_classes
        self.is_bidirectional = is_bidirectional

        self.encoder_emb = nn.Embedding(input_voc_size, self.input_size)
        self.encoder_lstm_layer = nn.LSTM(input_size=self.input_size, hidden_size=self.hidden_size, num_layers=self.lstm_layers, bidirectional=self.is_bidirectional, batch_first=True)

        self.decoder_emb = nn.Embedding(self.cnt_classes, self.input_size)
        self.decoder_lstm_layer = nn.LSTM(input_size=self.hidden_size, hidden_size=(1 + self.is_bidirectional)* self.hidden_size, num_layers=self.lstm_layers, batch_first = True)

        self.decoder_linear = nn.Linear(in_features=self.hidden_size, out_features=self.out_features)

    def forward(self, x, target_val, teacher_forcing_ratio=TEACHER_FORCING_RATIO):
        batch_n = x.shape[0] # сколько у нас слов в батче
        length_tensor = (x != PAD_IND).sum(axis=1).cpu() #длина последовательности индексов без паддинга

        x = self.encoder_emb(x)
        x = pack_padded_sequence(x, length_tensor, batch_first=True, enforce_sorted=False)

        _, prev_state = self.encoder_lstm_layer(x)

        prev_state = (prev_state[0].reshape(self.lstm_layers, batch_n, -1),
                      prev_state[1].reshape(self.lstm_layers, batch_n, -1))
        prev_token_idx = torch.tensor([SOS_IND] * batch_n).reshape(-1, 1).to(device)

        probs_tensor = None # сюда буде складывать результаты выходов decoder
        for i in range(NEC_TOKENS_LEN): # максимальная длина входной последовательности
            if i != 0 and target_val is not None and random.random() < teacher_forcing_ratio:
                prev_token_idx = target_val[:,i - 1].reshape(-1,1)

            prev_token_emb = self.decoder_emb(prev_token_idx)
            out, prev_state = self.decoder_lstm_layer(prev_token_emb, prev_state) # передаём состояние из encoder
            # out - распределения
            out = self.decoder_linear(out)
            prev_token_idx = out.argmax(axis=-1).reshape(batch_n, -1)

            if probs_tensor is None:
                probs_tensor = out.unsqueeze(0)
            else:
                probs_tensor = torch.cat([probs_tensor, out.unsqueeze(0)])

        return probs_tensor.transpose(1,0)


In [ ]:
train_dataset = TranslationDataset(x_seq_path=X_TRAIN_PATH, y_seq_path=Y_TRAIN_PATH, is_train=True, x_seq_len=NEC_TOKENS_LEN, y_seq_len=NEC_TOKENS_LEN)

val_dataset = TranslationDataset(x_seq_path=X_VALID_PATH, y_seq_path=Y_VALID_PATH, is_train=False, x_seq_len=NEC_TOKENS_LEN, y_seq_len=NEC_TOKENS_LEN)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

In [ ]:
if CNT_CLASSES is None:
  CNT_CLASSES = train_dataset.cnt_y_tokens()

In [ ]:
model = EncoderDecoderModel(input_voc_size=train_dataset.cnt_x_tokens(), device=device, cnt_classes=train_dataset.cnt_y_tokens())
model = model.to(device)

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=LR)

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
lambda_scheduler = lambda x: SCHEDULER_LAMBDA_PARAM ** x
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda_scheduler)

In [ ]:
def show_random_examples(input_data, out_answers, targets, dataloader):
    # show random texts with three ids
    for i in [1, 3, -1]:
        x_data = input_data[i]
        y_data = out_answers[i]
        target_data = targets[i]

        # get tokenizers
        x_tokenizer = dataloader.dataset.x_tokenizer
        y_tokenizer = dataloader.dataset.y_tokenizer

        # convert answers idx
        x_answers = x_tokenizer.convert_idx_to_tokens(x_data.cpu().detach().numpy().tolist())
        y_answers = y_tokenizer.convert_idx_to_tokens(y_data.cpu().detach().numpy().tolist())
        target_tokens = y_tokenizer.convert_idx_to_tokens(target_data.cpu().detach().numpy().tolist())

        print("X: ", ''.join([x for x in x_answers]))
        print("Answer: ", ''.join([y for y in y_answers]))
        print("Target: ", ''.join([trg for trg in target_tokens]))

def correct_tokens_for_bleu(tokens_list):
    res_tokens_list = []
    for tok in tokens_list:
      if tok == EOS_TOKEN:
        break
      res_tokens_list.append(tok)

    return [res_tokens_list]


def bleu_calculation(targets_ids, all_labels, tokenizer, bleu_metric):
    targets_ids = np.array(list(itertools.chain(*targets_ids))) # concat list by axis=0
    all_labels = np.array(list(itertools.chain(*all_labels)))

    targets_ids = targets_ids.reshape(-1, targets_ids.shape[-1])
    all_labels = all_labels.reshape(-1, all_labels.shape[-1])

    targets_list = [
        correct_tokens_for_bleu(
            tokenizer.convert_idx_to_tokens(trg_idx_list)
          )[0] for trg_idx_list in targets_ids]
    answers_list = [correct_tokens_for_bleu(
          tokenizer.convert_idx_to_tokens(label_list)
        ) for label_list in all_labels]

    targets_list = [''.join(t_list) for t_list in targets_list]
    answers_list = [[''.join(ref) for ref in answer] for answer in answers_list]

    return bleu_metric.compute(predictions=targets_list, references=answers_list)['bleu']


def train_one_epoch(epoch_index, train_loader, scheduler, optimizer, loss_fn, clip, teacher_forcing_ratio=TEACHER_FORCING_RATIO):
    running_loss = 0.0
    last_loss = 0.0
    train_loss = 0.0
    i = 0

    all_labels = []
    all_answers = []

    for i, data in tqdm(enumerate(train_loader)):
        input_data, labels = data
        input_data = input_data.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(input_data, labels, teacher_forcing_ratio=teacher_forcing_ratio)
        answers = outputs.argmax(axis=-1).squeeze(-1)

        loss = loss_fn(outputs.reshape(-1, CNT_CLASSES), labels[:, :outputs.shape[1]].reshape(-1))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        running_loss += loss.item()
        train_loss += loss.item()

        all_answers += [answers.cpu().detach().numpy().tolist()]
        all_labels += [labels.cpu().detach().numpy().tolist()]
        if i % WRITE_CNT_STEPS == WRITE_CNT_STEPS - 1 and epoch_index != -1:
            print("i =", i)
            # get answers idx by argmax
            show_random_examples(input_data, answers, labels, train_dataloader)

            # logging
            last_loss = running_loss / 100 # loss per batch
            tb_x = epoch_index * len(train_loader) + i + 1
            running_loss = 0.

            print("TRAIN LOSS =", last_loss)
            if i % SCHEDULER_CNT_STEPS == 0 and scheduler is not None:
                scheduler.step()
        i += 1
    bleu_score = bleu_calculation(all_answers, all_labels, train_loader.dataset.y_tokenizer, bleu_metric)
    print('TRAIN BLEU:', bleu_score)
    train_loss = train_loss / len(train_loader)
    return last_loss, bleu_score


def validation(epoch_number, val_dataloader):
    val_loss = 0.0
    all_answers = []
    all_labels = []

    # eval model mode
    model.eval()
    with torch.no_grad():
        for i, vdata in tqdm(enumerate(val_dataloader)):
            input_data, labels = vdata

            optimizer.zero_grad()

            # move data to device
            input_data = input_data.to(device)
            labels = labels.to(device).long()

            # model inference
            outputs = model(input_data, None)

            outputs_reshape = outputs.reshape(-1, CNT_CLASSES)

            # calc loss
            vloss = loss_fn(outputs_reshape, labels[:, :outputs.shape[1]].reshape(-1))
            val_loss += vloss.item()


            answers = outputs.argmax(axis=-1).squeeze(-1)
            all_answers += [answers.cpu().detach().numpy().tolist()]
            all_labels += [labels.cpu().detach().numpy().tolist()]
            # logging
            if i == 0:
                show_random_examples(input_data, answers, labels, val_dataloader)

    val_loss = val_loss / len(val_dataloader)

    bleu_score = bleu_calculation(
        all_answers,
        all_labels,
        val_dataloader.dataset.y_tokenizer,
        bleu_metric)

    print("VAL LOSS =", val_loss)
    return val_loss, bleu_score

In [ ]:
def train(
    model,
    model_name,
    train_dataloader,
    optimizer,
    scheduler, # could be None
    loss_fn,
    clip,
    teacher_forcing_ratio,
    epochs):
  best_vloss = 1_000_000.

  for epoch_number in tqdm(range(epochs)):
      print('EPOCH {}:'.format(epoch_number + 1))

      model.train(True)
      train_loss, train_bleu = train_one_epoch(
          epoch_index=epoch_number,
          train_loader=train_dataloader,
          optimizer=optimizer,
          scheduler=scheduler,
          loss_fn=loss_fn,
          clip=clip,
          teacher_forcing_ratio=teacher_forcing_ratio)

      model.train(False)
      val_loss, val_bleu = validation(epoch_number, val_dataloader)

      # logging
      # save models if val_loss is better than best_vloss
      if val_loss < best_vloss:
          best_vloss = val_loss
          model_path = os.path.join(model_name, 'model_{}_{}'.format(epoch_number + 1, timestamp))
          torch.save(model.state_dict(), model_path)

  return model


In [ ]:
TRAIN = True
if TRAIN:
  base_enc_dec_model = train(
      model=model,
      model_name=MODEL_NAME,
      train_dataloader=train_dataloader,
      optimizer=optimizer,
      scheduler=scheduler,
      loss_fn=loss_fn,
      clip=CLIP,
      teacher_forcing_ratio=TEACHER_FORCING_RATIO,
      epochs=EPOCHS
  )

  0%|          | 0/10 [00:00<?, ?it/s]

EPOCH 1:


0it [00:00, ?it/s]

i = 99
X:  <SOS>it is a 15minute walk the knigsallee, the old town and the river rhine.<EOS><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD>
Answer:  <SOS>в                                                     <PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD>
Target:  <SOS>прогулка до бульвара кёнигсаллее, старого города и реки рейн занимает 15 минут.<EOS><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD>
X:  <SOS>some rooms have skylights.<EOS><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD

0it [00:00, ?it/s]

ValueError: too many values to unpack (expected 2)

Теперь Teacher Forcing

In [ ]:
MODEL_NAME = 'lstm_teacher_forcing_model'
TEACHER_FORCING_RATIO = 0.5

if not os.path.exists(MODEL_NAME):
  os.makedirs(MODEL_NAME)

# define model and move it to device
model = EncoderDecoderModel(
    input_voc_size=train_dataset.cnt_x_tokens(),
    device=device,
    cnt_classes=train_dataset.cnt_y_tokens())
model = model.to(device)

# writer
writer = SummaryWriter('runs/{}'.format(MODEL_NAME))

# redefine optimizer
optimizer = Adam(model.parameters(), lr=LR)

In [ ]:
if TRAIN:
  teacher_forcing_enc_dec_model = train(
      model=model,
      model_name=MODEL_NAME,
      train_dataloader=train_dataloader,
      optimizer=optimizer,
      scheduler=scheduler,
      loss_fn=loss_fn,
      clip=CLIP,
      teacher_forcing_ratio=TEACHER_FORCING_RATIO,
      epochs=EPOCHS
  )

  0%|          | 0/10 [00:00<?, ?it/s]

EPOCH 1:


0it [00:00, ?it/s]

i = 99
X:  <SOS>an internet corner is available free of charge.<EOS><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD>
Answer:  <SOS>в                                               <PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD>
Target:  <SOS>в распоряжении гостей бесплатный интернетуголок.<EOS><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD>
X:  <SOS>guests also enjoy free wifi acc

0it [00:00, ?it/s]

ValueError: too many values to unpack (expected 2)

Теперь поработаем с attention

In [1]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, temperature, attn_dropout=0.1):
        super().__init__()
        self.temperature = temperature
        self.dropout = nn.Dropout(attn_dropout)

    def forward(self, q, k, v, mask=None):
        attn = torch.matmul(q / self.temperature, k.transpose(2, 3))

        if mask is not None:
            attn = attn.masked_fill(mask == 0, -1e9)

        attn = self.dropout(torch.nn.functional.softmax(attn, dim=-1))
        output = torch.matmul(attn, v)

        return output, attn

class Attention(nn.Module):
    def __init__(self, d_model, d_k, d_v, dropout=0.1, n_head=1):
        super(Attention, self).__init__()
        self.n_head = n_head
        self.d_k = d_k
        self.d_v = d_v
        self.dropout = dropout
        self.W_k = nn.Linear(in_features=d_model, out_features=d_k * n_head, bias=False)
        self.W_q = nn.Linear(in_features=d_model, out_features=d_k * n_head, bias=False)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_v * n_head, bias=False)

        self.dot_product_attention = ScaledDotProductAttention(
            temperature=np.sqrt(d_k),
            attn_dropout=self.dropout
          )

    def forward(self, q, k, v, mask=None, return_attention=False):
        d_k, d_v, n_head = self.d_k, self.d_v, self.n_head
        sz_b, len_q, len_k, len_v = q.size(0), q.size(1), k.size(1), v.size(1)

        q = self.W_q(q).view(sz_b, len_q, n_head, d_k)
        k = self.W_k(k).view(sz_b, len_k, n_head, d_k)
        v = self.W_v(v).view(sz_b, len_v, n_head, d_v)

        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

        if mask is not None:
            mask = mask.unsqueeze(1)

        q, attn = self.dot_product_attention(q, k, v, mask=mask)
        if return_attention:
          return q, attn
        return q

NameError: name 'nn' is not defined

In [ ]:
class EncoderDecoderAttentionModel(nn.Module):
    # Simple Encoder-Decoder Model with 3 GRU Layers in encoder
    # And with 3 GRU in decoder, forward could be run with teacher_forcing
    def __init__(self,
                 input_voc_size,
                 cnt_classes,
                 device=None,
                 input_size=INPUT_SIZE,
                 hidden_size=HIDDEN_SIZE,
                 lstm_layers=LSTM_LAYERS,
                 is_bidirectional=IS_BIDIRECTIONAL_LAYERS):
        super(EncoderDecoderAttentionModel, self).__init__()

        self.input_size = input_size
        self.lstm_layers = lstm_layers
        self.cnt_classes = cnt_classes
        self.hidden_size = hidden_size
        self.out_features = 1 if self.cnt_classes == 2 else self.cnt_classes
        self.is_bidirectional = is_bidirectional

        self.encoder_embed = nn.Embedding(input_voc_size, self.input_size)
        self.encoder_lstm_layer = nn.LSTM(
            input_size=self.input_size,
            hidden_size=self.hidden_size,
            num_layers=self.lstm_layers,
            bidirectional=self.is_bidirectional,
            batch_first=True
        )
        self.attention = Attention(
            d_model=self.hidden_size,
            d_k=self.hidden_size,
            d_v=self.hidden_size,
            dropout=0
        )

        self.decoder_embed = nn.Embedding(self.cnt_classes, self.input_size)
        self.decoder_lstm_layer = nn.LSTM(
            input_size=2 * self.hidden_size, # concat attention and decoder hidden
            hidden_size=(1 + self.is_bidirectional) * self.hidden_size,
            num_layers=self.lstm_layers,
            batch_first=True
        )
        self.decoder_linear = nn.Linear(
            in_features=self.hidden_size,
            out_features=self.out_features)

    def forward(
            self,
            x,
            target_val,
            teacher_forcing_ratio = TEACHER_FORCING_RATIO):
        batch_n = x.shape[0]
        length_tensor = (x != PAD_IND).sum(axis=1) # get seq lens to pack

        ### encoder
        encod_emb = self.encoder_embed(x)
        # pack sequence
        packed_emb = pack_padded_sequence(encod_emb, length_tensor, batch_first=True, enforce_sorted=False)
        # encoder inference
        packed_out, enc_prev_state = self.encoder_lstm_layer(packed_emb)
        # enc_hiddens - hiddens for each iteration for last lstm layer
        enc_hiddens, _ = pad_packed_sequence(packed_out, batch_first=True)

        # reshape prev_state
        enc_prev_state = (enc_prev_state[0].reshape(self.lstm_layers, batch_n, -1),
                      enc_prev_state[1].reshape(self.lstm_layers, batch_n, -1))
        prev_state = enc_prev_state
        # create <sos> tokens for each sentence
        prev_token_idx = torch.tensor([SOS_IND] * batch_n).reshape(-1, 1).to(device)

        # create probs for each token
        probs_tensor = None
        for i in range(NEC_TOKENS_LEN):
            if i != 0 and target_val is not None and random.random() < teacher_forcing_ratio:
                # teacher forcing
                prev_token_idx = target_val[:, i - 1].reshape(-1, 1)

            # decoder step
            prev_token_emb = self.decoder_embed(prev_token_idx)
            # sum of decoder layers hiddens of iteration
            prev_dec_hidden = prev_state[0].sum(axis=0).unsqueeze(1)

            # attention between decoder and encoder
            out, _ = self.attention(prev_dec_hidden, enc_hiddens, enc_hiddens, return_attention=True)
            out = out.sum(axis=1) # sum for different attention heads

            # concat attention result and embeding of prev token
            decoder_input_embed = torch.cat([prev_token_emb, out], axis=-1)
            out, prev_state = self.decoder_lstm_layer(decoder_input_embed, prev_state)
            out = self.decoder_linear(out)

            # token idx answer by greedy strategy
            prev_token_idx = out.argmax(axis=-1).reshape(batch_n, -1)

            # save probs for answers
            if probs_tensor is None:
                probs_tensor = out.unsqueeze(0)
            else:
                probs_tensor = torch.cat([probs_tensor, out.unsqueeze(0)])

        return probs_tensor.transpose(1, 0)

In [ ]:
MODEL_NAME = 'lstm_attention_model'
TEACHER_FORCING_RATIO = 0
TRAIN = True

if not os.path.exists(MODEL_NAME):
  os.makedirs(MODEL_NAME)

# define model and move it to device
model = EncoderDecoderAttentionModel(
    input_voc_size=train_dataset.cnt_x_tokens(),
    device=device,
    cnt_classes=train_dataset.cnt_y_tokens())
model = model.to(device)

# writer
writer = SummaryWriter('runs/{}'.format(MODEL_NAME))

# redefine optimizer
optimizer = Adam(model.parameters(), lr=LR)
if TRAIN:
  attention_enc_dec_model = train(
      model=model,
      model_name=MODEL_NAME,
      train_dataloader=train_dataloader,
      optimizer=optimizer,
      scheduler=scheduler,
      loss_fn=loss_fn,
      writer=writer,
      clip=CLIP,
      teacher_forcing_ratio=TEACHER_FORCING_RATIO,
      epochs=EPOCHS
  )